In [11]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import load_model


In [12]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load
labels_file = ROOT_DIR / "datasets" / "labels.csv"
model_file  = ROOT_DIR / "src" / "model_training" / "checkpoints" / "final_model.h5"
results_dir = ROOT_DIR / "src" / "model_training" / "results"
results_dir.mkdir(exist_ok=True)
output_file = results_dir / "model_predictions.csv"

In [13]:
model = load_model(model_file)
print("Model input shape:",model.input_shape)

c:\Users\mayak\anaconda3\envs\shapenv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model input shape: (None, 7060)


In [14]:
train_features, val_features, all_train_features, test_features, train_targets, val_targets, all_train_targets, test_targets = load(norm='norm')
print("Train features shape:", train_features.shape)
print("All train features shape:", all_train_features.shape)
print("Test features shape:", test_features.shape)
all_features = np.vstack([all_train_features, test_features])
print("All features shape:", all_features.shape)

Train features shape: (13884, 7060)
All train features shape: (18498, 7060)
Test features shape: (4554, 7060)
All features shape: (23052, 7060)


In [15]:
all_predictions = model.predict(all_features, batch_size=1024).squeeze()
print("Predictions shape:", all_predictions.shape)

23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 75ms/step
Predictions shape: (23052,)


In [16]:
labels_df = pd.read_csv(labels_file)
labels_train = labels_df[labels_df['fold'] != 0]
labels_test = labels_df[labels_df['fold'] == 0]
labels_ordered = pd.concat([labels_train, labels_test], axis=0).reset_index(drop=True)

In [18]:
labels_ordered['ID']=(
    labels_ordered['drug_a_name'] + "_" +
    labels_ordered['drug_b_name'] + "_" +
    labels_ordered['cell_line'])
labels_ordered['predicted_synergy'] = all_predictions.round(8)
pred_df = labels_ordered[['ID', 'drug_a_name', 'drug_b_name', 'cell_line', 'synergy', 'predicted_synergy']]

In [19]:
pred_df.to_csv(output_file,index=False)
print(f"Predictions saved to: {output_file}")
print(pred_df.head())

Predictions saved to: c:\Users\mayak\DeepSyn\src\model_training\results\model_predictions.csv
                   ID drug_a_name drug_b_name cell_line   synergy  \
0  5-FU_ABT-888_A2058        5-FU     ABT-888     A2058  7.693530   
1  5-FU_ABT-888_A2780        5-FU     ABT-888     A2780  7.778053   
2   5-FU_ABT-888_A375        5-FU     ABT-888      A375 -1.198505   
3   5-FU_ABT-888_A427        5-FU     ABT-888      A427  2.595684   
4  5-FU_ABT-888_CAOV3        5-FU     ABT-888     CAOV3 -5.139971   

   predicted_synergy  
0           0.415782  
1           5.999769  
2           0.504890  
3           0.415782  
4           0.415782  
